1. Install the Transformers library.
2. Load the pre-trained T5 model and the associated tokenizer.
3. Prepare your dataset: tokenize the inputs and the labels.
4. Train the model on your dataset.
5. Evaluate the model on your validation set.

In [1]:
import torch
torch.cuda.empty_cache()

In [ ]:
!pip install simplet5

In [3]:
from simplet5 import SimpleT5

2024-02-09 22:45:59.352742: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-09 22:45:59.352833: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-09 22:45:59.473293: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
model = SimpleT5()
model.from_pretrained("t5","t5-base")

Downloading:   0%|          | 0.00/773k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/850M [00:00<?, ?B/s]

In [5]:
import pandas as pd

In [6]:
train_df = pd.read_csv("/kaggle/input/trainingdata/trainingdata.csv")
train_df.head()

,source_text,target_text
0,US consumer spending dropped by a record in Ap...,consumer spending plunges 13.6 percent in Apri...
1,State-run lenders require an urgent Rs 1.2 tri...,government will have to take a bulk of the tab...
2,Apparel exporters on Wednesday urged the gover...,exporters are facing issues in terms of raw ma...
3,Asian shares battled to extend a global reboun...,the dollar loses some ground on the safe haven...
4,After Indiaâ€™s sovereign credit rating fell t...,six Indian public-sector undertakings have tak...


In [7]:
train_df['source_text'][4]

'After Indiaâ€™s sovereign credit rating fell to lowest investment grade â€˜Baa3â€™ early in June, six Indian public-sector undertakings (PSUs) have also taken a hit with them now becoming potential â€œfallen angelsâ€\x9d. These six companies in the non-financial sector viz â€” Indian Oil Corporation, Hindustan Petroleum Corporation, Oil India, Petronet LNG, Bharat Petroleum Corporation and Oil and Natural Gas Corporation â€” are now just one step away from being considered junk, global ratings agency Moodyâ€™s Investors Service said on Tuesday. â€œFallen Angelsâ€\x9d are those companies which were earlier in the investment grade category and have slipped to sub-investment grade.\n\nAs the coronavirus pandemic has afflicted markets and economy around the world, the Asian list of potential â€œfallen angelsâ€\x9d has reached an all time high of 21 in early June owing to the addition of six Indian PSUs, the agency said. The quantum of names in the list has doubled due to the COVID-19 pand

In [8]:
# changing all values to string type
train_df['source_text'] = train_df['source_text'].astype(str)
# replacing \n by ''
train_df['source_text'] = train_df['source_text'].str.replace('\n','')
# removing garbage characters using regex
train_df['source_text'] = train_df['source_text'].str.replace(r'[^a-zA-Z0-9\s.,-]', '', regex=True)
train_df['source_text'][4]

'After Indias sovereign credit rating fell to lowest investment grade Baa3 early in June, six Indian public-sector undertakings PSUs have also taken a hit with them now becoming potential fallen angels. These six companies in the non-financial sector viz  Indian Oil Corporation, Hindustan Petroleum Corporation, Oil India, Petronet LNG, Bharat Petroleum Corporation and Oil and Natural Gas Corporation  are now just one step away from being considered junk, global ratings agency Moodys Investors Service said on Tuesday. Fallen Angels are those companies which were earlier in the investment grade category and have slipped to sub-investment grade.As the coronavirus pandemic has afflicted markets and economy around the world, the Asian list of potential fallen angels has reached an all time high of 21 in early June owing to the addition of six Indian PSUs, the agency said. The quantum of names in the list has doubled due to the COVID-19 pandemic. The 21 Asian companies, which now risk becomi

In [9]:
# changing all values to string type
train_df['target_text'] = train_df['target_text'].astype(str)

In [10]:
train_df['target_text'][0]

'consumer spending plunges 13.6 percent in April. that was the biggest drop since the government started tracking series in 1959. consumer spending accounts for more than two-thirds of economic activity. economists polled by Reuters had forecast consumer spending plummeting 12.6 percent. a spokesman for the u.s. government said the data was not available.'

In [11]:
# changing first word of each sentence to upper case
train_df['target_text'] = train_df['target_text'].apply(lambda x: '. '.join(sentence[0].upper()+sentence[1:] for sentence in x.split('. ') if sentence!=''))
train_df['target_text'][0]

'Consumer spending plunges 13.6 percent in April. That was the biggest drop since the government started tracking series in 1959. Consumer spending accounts for more than two-thirds of economic activity. Economists polled by Reuters had forecast consumer spending plummeting 12.6 percent. A spokesman for the u.s. Government said the data was not available.'

In [12]:
train_df.head()

,source_text,target_text
0,US consumer spending dropped by a record in Ap...,Consumer spending plunges 13.6 percent in Apri...
1,State-run lenders require an urgent Rs 1.2 tri...,Government will have to take a bulk of the tab...
2,Apparel exporters on Wednesday urged the gover...,Exporters are facing issues in terms of raw ma...
3,Asian shares battled to extend a global reboun...,The dollar loses some ground on the safe haven...
4,After Indias sovereign credit rating fell to l...,Six Indian public-sector undertakings have tak...


In [13]:
# load train_df to datasetdict
from datasets import Dataset, DatasetDict
dataset = Dataset.from_pandas(train_df)
dataset = DatasetDict({'train': dataset})
dataset

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text'],
        num_rows: 33365
    })
})

In [14]:
# split the data into train and test
from sklearn.model_selection import train_test_split
splitted_data = dataset['train'].train_test_split(test_size=0.2)
dataset = DatasetDict({
    'train': splitted_data['train'],
    'test': splitted_data['test']
})

In [ ]:
!pip install transformers

In [16]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("t5-base")

In [17]:
tokenizer("Hello, this one sentence!")

{'input_ids': [8774, 6, 48, 80, 7142, 55, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [18]:
tokenizer(["Hello, this one sentence!", "This is another sentence."])

{'input_ids': [[8774, 6, 48, 80, 7142, 55, 1], [100, 19, 430, 7142, 5, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [19]:
with tokenizer.as_target_tokenizer():
    print(tokenizer(["Hello, this one sentence!", "This is another sentence."]))

{'input_ids': [[8774, 6, 48, 80, 7142, 55, 1], [100, 19, 430, 7142, 5, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


In [20]:
prefix = "summarize: "

In [21]:
max_input_length = 1024
max_target_length = 128

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["source_text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [22]:
preprocess_function(dataset['train'][:2])

{'input_ids': [[21603, 10, 619, 3, 115, 7, 15, 619, 29, 7, 15, 619, 14816, 1960, 7, 301, 566, 1537, 419, 9928, 565, 18080, 3937, 391, 3502, 11636, 11, 3, 16143, 2578, 8530, 4159, 7735, 107, 736, 3478, 23, 30, 2083, 489, 243, 419, 9928, 565, 7802, 32, 65, 582, 8, 2015, 12024, 16, 8, 1125, 628, 16, 1244, 20008, 5, 134, 14661, 53, 44, 8, 1360, 20008, 3699, 1769, 12968, 6, 736, 3478, 23, 243, 20008, 47, 8, 3, 2935, 26, 109, 13, 2913, 2121, 16, 1547, 147, 3, 9, 2646, 977, 5, 852, 6, 8, 97, 65, 369, 21, 39, 538, 12, 582, 3, 9, 2488, 12, 8, 21670, 9199, 12197, 5, 683, 23, 32, 141, 3, 10671, 91, 296, 18, 4057, 3620, 28, 20043, 13, 2950, 364, 12, 199, 3607, 9284, 13, 8, 2896, 13, 8, 21670, 9199, 12197, 6, 736, 3478, 23, 243, 5, 8123, 3478, 23, 12566, 24, 419, 9928, 2319, 1729, 13, 11715, 204, 16235, 23095, 16, 1244, 20008, 3744, 12, 81, 80, 18, 324, 189, 13, 8, 975, 24422, 6203, 792, 7686, 16, 1547, 78, 623, 5, 391, 3502, 7, 1729, 16, 8, 538, 47, 11715, 6464, 2560, 23095, 16, 5123, 3, 88, 243, 

In [23]:
tokenized_datasets = dataset.map(preprocess_function, batched=True)

  0%|          | 0/27 [00:00<?, ?ba/s]

  0%|          | 0/7 [00:00<?, ?ba/s]

In [24]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

In [25]:
batch_size = 4
args = Seq2SeqTrainingArguments(
    "test-summarization",
    evaluation_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=True,
)

In [26]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
!pip install rouge_score

In [28]:
from datasets import load_metric
metric = load_metric("rouge")

In [29]:
import nltk
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # Extract a few results
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}

    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [30]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Using amp half precision backend


In [31]:
trainer.train()

The following columns in the training set  don't have a corresponding argument in `T5ForConditionalGeneration.forward` and have been ignored: source_text, target_text.
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use thePyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 26692
  Num Epochs = 1
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 4
  Gradient Accumulation steps = 1
  Total optimization steps = 6673
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/a

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,0.605100,0.467429,35.326200,16.276500,33.715700,34.774600,16.484200


Saving model checkpoint to test-summarization/checkpoint-500
Configuration saved in test-summarization/checkpoint-500/config.json
Model weights saved in test-summarization/checkpoint-500/pytorch_model.bin
tokenizer config file saved in test-summarization/checkpoint-500/tokenizer_config.json
Special tokens file saved in test-summarization/checkpoint-500/special_tokens_map.json
Copy vocab file to test-summarization/checkpoint-500/spiece.model
Saving model checkpoint to test-summarization/checkpoint-1000
Configuration saved in test-summarization/checkpoint-1000/config.json
Model weights saved in test-summarization/checkpoint-1000/pytorch_model.bin
tokenizer config file saved in test-summarization/checkpoint-1000/tokenizer_config.json
Special tokens file saved in test-summarization/checkpoint-1000/special_tokens_map.json
Copy vocab file to test-summarization/checkpoint-1000/spiece.model
Saving model checkpoint to test-summarization/checkpoint-1500
Configuration saved in test-summarization/

TrainOutput(global_step=6673, training_loss=0.6216298976116732, metrics={'train_runtime': 6255.6124, 'train_samples_per_second': 4.267, 'train_steps_per_second': 1.067, 'total_flos': 2.905564552077312e+16, 'train_loss': 0.6216298976116732, 'epoch': 1.0})

In [34]:
# save the finetuned model
trainer.save_model("t5-base-finetuned")

Saving model checkpoint to t5-base-finetuned
Configuration saved in t5-base-finetuned/config.json
Model weights saved in t5-base-finetuned/pytorch_model.bin
tokenizer config file saved in t5-base-finetuned/tokenizer_config.json
Special tokens file saved in t5-base-finetuned/special_tokens_map.json
Copy vocab file to t5-base-finetuned/spiece.model


In [35]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the saved model and tokenizer
model = T5ForConditionalGeneration.from_pretrained('/kaggle/working/t5-base-finetuned')
tokenizer = T5Tokenizer.from_pretrained('/kaggle/working/t5-base-finetuned')

# Prepare your input data
input_text = "After more than two weeks of testimony, the jury at the coroner's inquest into the stabbing massacre at James Smith Cree Nation shared its recommendations to help prevent similar tragedies in the future. The inquest, which began on Jan. 15 in Melfort, Sask., has been examining the events of Sept. 4, 2022, when Myles Sanderson killed 11 people — 10 from James Smith and one from the neighbouring village of Weldon. During the past two weeks, jurors have heard from 30 witnesses including RCMP officers, a psychologist, forensic pathologists, Sanderson's former partner, officials from the Parole Board of Canada and the elders who worked with Sanderson in prison."
inputs = tokenizer(input_text, return_tensors='pt')

# Generate predictions
outputs = model.generate(inputs['input_ids'])

# Decode the predictions
predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
predicted_text

loading configuration file /kaggle/working/t5-base-finetuned/config.json
Model config T5Config {
  "_name_or_path": "t5-base",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_num_buckets": 32,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": true,
      "max_length": 300,
      "num_beams": 4,
      "prefix": "

"The jury at the coroner's inquest into the stabbing massacre at James"